In [2]:
import random
from tqdm import tqdm
import folium
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(
    { "figure.figsize": (17, 7) },
    style='ticks',
    palette=sns.color_palette("Set2"),
    color_codes=True,
    font_scale=5
)

plt.rcParams.update({
    "axes.labelsize": 12,  # Axes label font size
})

%config InlineBackend.figure_format = 'retina'
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load stops
stops_df = pd.read_csv("timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\stops.txt")

# Isolate Bradford
stops_df = stops_df.loc[(stops_df["stop_lat"] < 54) & (stops_df["stop_lat"] > 53.7)].reset_index(drop=True)
stops_df = stops_df.loc[(stops_df["stop_lon"] < -1.55) & (stops_df["stop_lon"] > -1.95)].reset_index(drop=True)

In [4]:
def get_stop_prox_order(origin:tuple):
    return list(stops_df.loc[(abs(stops_df["stop_lat"] - origin[0]) + abs(stops_df["stop_lon"] - origin[1])).sort_values(ascending=True).index, "stop_id"])

# Get routes through each stop

In [5]:
stop_times_df = pd.read_csv("timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\stop_times.txt")
trips = pd.read_csv("timetables-20260312\\var\\www\\data.datalibrary.uk\\transport\\BODS-ARCHIVE\\timetables\\2026\\03\\12\\itm_yorkshire_gtfs_20260312\\trips.txt")

stop_times_df = stop_times_df.merge(
    trips[["route_id", "trip_id"]],
    left_on='trip_id', 
    right_on='trip_id', 
    how='left')

stop_times_df["arrival_time"] = pd.to_timedelta(stop_times_df["arrival_time"])
stop_times_df["departure_time"] = pd.to_timedelta(stop_times_df["departure_time"])

In [ ]:
stop_route_dict = stop_times_df.groupby("stop_id")["route_id"].unique().to_dict()

# Find route

In [309]:
start_point = (53.79,-1.76)
end_point = (53.841401, -1.827540)

In [310]:
org_route_pref_order = []
route_best_start_stops = []

dest_route_pref_order = []
route_best_end_stops = []

stop_prox_order = get_stop_prox_order(start_point)

for i in range(len(stop_prox_order)):
    try:
        new_routes = list(stop_route_dict[stop_prox_order[i]])
        org_route_pref_order += new_routes
        route_best_start_stops += [stop_prox_order[i]] * len(new_routes)
    except:
        print(f"No routes found at: i: {i}, cur stop id: {stop_prox_order[i]}")


stop_prox_order = get_stop_prox_order(end_point)

for i in range(len(stop_prox_order)):
    try:
        new_routes = list(stop_route_dict[stop_prox_order[i]])
        dest_route_pref_order += new_routes
        route_best_end_stops += [stop_prox_order[i]] * len(new_routes)
    except:
        print(f"No routes found at: i: {i}, cur stop id: {stop_prox_order[i]}")

No routes found at: i: 72, cur stop id: 450G9423
No routes found at: i: 2135, cur stop id: 450G7922
No routes found at: i: 2310, cur stop id: 450G9178
No routes found at: i: 2455, cur stop id: 450G7949
No routes found at: i: 3782, cur stop id: 450G7923
No routes found at: i: 4467, cur stop id: 450G7920
No routes found at: i: 4952, cur stop id: 450G7935
No routes found at: i: 5169, cur stop id: 450G6011
No routes found at: i: 5959, cur stop id: 450G9566
No routes found at: i: 886, cur stop id: 450G7935
No routes found at: i: 1371, cur stop id: 450G9566
No routes found at: i: 2075, cur stop id: 450G9423
No routes found at: i: 2730, cur stop id: 450G7923
No routes found at: i: 3949, cur stop id: 450G9178
No routes found at: i: 4310, cur stop id: 450G7920
No routes found at: i: 4517, cur stop id: 450G7922
No routes found at: i: 5063, cur stop id: 450G7949
No routes found at: i: 6426, cur stop id: 450G6011


In [311]:
min_sum = np.inf
best_route_id = None
best_start_index = None
best_end_index = None

for i, route_id in enumerate(org_route_pref_order):
    idx_sum = i + dest_route_pref_order.index(route_id)
    if idx_sum < min_sum:
        min_sum = idx_sum
        best_route_id = route_id
        best_start_index = i
        best_end_index = dest_route_pref_order.index(route_id)
        print(f"new_low! i:{idx_sum}, rid:{route_id}, sum:{idx_sum}")

new_low! i:3490, rid:12758, sum:3490
new_low! i:3158, rid:12707, sum:3158
new_low! i:1898, rid:32596, sum:1898
new_low! i:994, rid:12628, sum:994
new_low! i:73, rid:12770, sum:73
new_low! i:23, rid:12777, sum:23


In [312]:
initial_stop = route_best_start_stops[best_start_index]
end_stop = route_best_end_stops[best_end_index]
initial_stop, end_stop

('450023176', '450021124')

In [314]:
route_df = stop_times_df[stop_times_df["route_id"] == best_route_id]

trips_with_stops = route_df[route_df["stop_id"].isin([initial_stop, end_stop])]
trips_with_both = trips_with_stops.groupby("trip_id").filter(lambda x: x["stop_id"].nunique() == 2)["trip_id"].unique()

valid_trips_df = route_df[route_df["trip_id"].isin(trips_with_both)]
max_sequences = valid_trips_df.groupby("trip_id")["stop_sequence"].max()

longest_trip_id = max_sequences.idxmax()
max_trip_len = max_sequences.max()

print(f"Longest Trip ID: {longest_trip_id} with {max_trip_len} stops.")

sample_journey = stop_times_df.loc[stop_times_df["trip_id"] == longest_trip_id]

start_seq_id = sample_journey.loc[sample_journey["stop_id"] == initial_stop, "stop_sequence"].item()
end_seq_id = sample_journey.loc[sample_journey["stop_id"] == end_stop, "stop_sequence"].item()

Longest Trip ID: VJ04301f046f17787c0f86add58c667022792adb85 with 74 stops.


In [315]:
# trips_through_start = stop_times_df.loc[(stop_times_df["stop_id"] == initial_stop) & (stop_times_df["route_id"] == best_route_id)]
# trips_through_end = stop_times_df.loc[(stop_times_df["stop_id"] == end_stop) & (stop_times_df["route_id"] == best_route_id)] 

# trip_id = (set(trips_through_start["trip_id"].unique()) & set(trips_through_end["trip_id"].unique())).pop()
# sample_journey = stop_times_df.loc[stop_times_df["trip_id"] == trip_id]

# start_seq_id = sample_journey.loc[sample_journey["stop_id"] == initial_stop, "stop_sequence"].item()
# end_seq_id = sample_journey.loc[sample_journey["stop_id"] == end_stop, "stop_sequence"].item()

In [316]:
m = folium.Map(location=[53.79, -1.75], zoom_start=11, tiles='CartoDB positron')

folium.CircleMarker(
    location=start_point,
    radius=3,
    popup=folium.Popup("start", max_width=300),
    color="red",
    weight=1,
    fill=True,
    fill_color="red",
    fill_opacity=0.8
).add_to(m)

folium.CircleMarker(
    location=end_point,
    radius=3,
    popup=folium.Popup("end", max_width=300),
    color="green",
    weight=1,
    fill=True,
    fill_color="green",
    fill_opacity=0.8
).add_to(m)

sample_journey = stop_times_df.loc[stop_times_df["trip_id"] == longest_trip_id].reset_index()
for i in range(len(sample_journey)):
    stop_id = sample_journey.loc[i, "stop_id"]
    seq_id = sample_journey.loc[i, "stop_sequence"]
    if seq_id < start_seq_id or seq_id > end_seq_id:
        continue
    try:
        lat = stops_df.loc[stops_df["stop_id"] == stop_id, "stop_lat"].item()
        lon = stops_df.loc[stops_df["stop_id"] == stop_id, "stop_lon"].item()
    except:
        continue
    spot_col = "yellow"
    if stop_id == initial_stop or stop_id == end_stop:
        spot_col = "blue"
    folium.CircleMarker(
            location=[lat, lon],
            radius=3,
            color=spot_col,
            weight=1,
            fill=True,
            fill_color=spot_col,
            fill_opacity=0.3
        ).add_to(m)
    
m.save("route_map.html")